## ARIMA (CSS) Model

This tutorial explains how to use the ARIMA(p, d, q) model implemented in JAX.
This implementation estimates ARIMA parameters via **Conditional Sum-of-Squares (CSS)** on the differenced ("working") series.

In [3]:
import sys
import types
from pathlib import Path

# Add the project root to path so the package imports resolve from either the repo root or this folder
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "chronax").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
sys.modules.setdefault("chronax.utils.plotting", types.ModuleType("chronax.utils.plotting"))

import jax
import jax.numpy as jnp
import numpy as np

from chronax.models import ARIMA

# Model Overview

ARIMA models a time series using three components:

- **AR(p)**: autoregressive dependence on the past *p* values
- **I(d)**: differencing *d* times to remove non-stationarity
- **MA(q)**: moving-average dependence on the past *q* forecast errors

Parameters are estimated by minimizing the CSS objective (sum of squared residuals).

# Math Overview

## Differencing (I)
If d = 0, $y_t = Y_t$

If d = 1: $y_t = Y_t - Y_{t-1}$

If d = 2: $y_t = (Y_t - Y_{t-1}) - (Y_{t-1} - Y_{t-2})$

## ARMA
For $t \geq m = \max(p, q)$:

$$\hat{w}_t = c + \sum_{i=1}^{p}\phi_i w_{t-i} + \sum_{j=1}^{q}\theta_j e_{t-j}$$

Residual:

$$e_t = w_t - \hat{w}_t$$

## CSS Objective

CSS minimizes the sum of squared residuals for $t \geq m$:

$$\min_{\phi, \theta, c} \sum_{t=m}^{n-1} e_t^2$$

## Inverting Difference
Forecasts are produced on the working scale and then converted back to the original scale by exact discrete integration.

# CSS vs Kalman/Exact MLE

This ARIMA uses CSS, not a Kalman filter / exact Gaussian MLE.

**Consequences:**

Parameters can differ from statsmodels statespace ARIMA even with the same order.

**Differences are more likely with:**
* Nontrivial MA terms
* Short samples
* Rough objectives / local minima

# When to Use
**Use when:**
* You need a strong classical baseline
* Series becomes stationary after differencing
* You want interpretable AR/MA structure

**Avoid when:**
* Strong seasonal structure exists and you are not using `seasonal_order` / `period`
* Extremely short series (CSS becomes unstable)

# API Contract

## Constructor

```python
ARIMA(
    order: Tuple[int, int, int] = (0, 0, 0),
    seasonal_order: Tuple[int, int, int] = (0, 0, 0),
    period: int = 1,
    include_mean: bool = True,
    method: str = "CSS",
    alias: str = "ARIMA",
    standardize: bool = True,
)
```

**Parameters:**
- `order`: (p, d, q) where p = AR order, d = differencing order, q = MA order
- `seasonal_order`: seasonal (P, D, Q) order; use `(0, 0, 0)` for non-seasonal ARIMA
- `period`: seasonal frequency used with `seasonal_order`
- `include_mean`: Whether to include an intercept or drift term
- `method`: optimization objective, typically `"CSS"`, `"ML"`, or `"CSS-ML"`
- `alias`: Display name for the model
- `standardize`: Standardize the series before fitting for numerical stability

## Fit and Predict

### Fit Method

```python
fit(y: ArrayLike, X: Optional[ArrayLike] = None) -> ARIMA
```
- `y`: Observed time series
- `X`: Optional exogenous regressor matrix aligned with `y`

**Results of fit():** `self.model_` stores:
- `"arma"`: expanded ARIMA metadata `(p, q, P, Q, period, d, D)`
- `"coef"`: fitted parameter vector (AR/MA terms and optional mean/drift)
- `"sigma2"`: CSS residual variance estimate
- `"residuals"`: residual series on the working scale
- `"innovations"`: state-space innovations used for forecasting corrections
- `"success"`: whether the optimization converged to a finite fit

### Predict Method

```python
predict(h: int, X: Optional[ArrayLike] = None, level: Optional[int | Tuple[int, ...]] = None) -> Dict[str, ArrayLike]
```
- `h`: forecast horizon
- `level`: optional confidence level or tuple of levels for Gaussian prediction intervals

**Returns:** Dictionary with `"mean"` (shape `(h,)`) and optionally `"lo-{level}"`, `"hi-{level}"`.

### Forecast (Stateless)

```python
forecast(h: int, y: ArrayLike, X: Optional[ArrayLike] = None) -> Dict[str, ArrayLike]
```
Stateless variant that fits and predicts without mutating `self.model_`. It returns only `"mean"`.

# Examples

## Fit and Predict with Differencing

We fit ARIMA(1,1,1) on trended data and verify the forecasts are finite after inverting the differencing.

In [ ]:
# Helper: reproducible random data
def _randn(n, scale=1.0, seed=0):
    key = jax.random.PRNGKey(seed)
    return scale * jax.random.normal(key, shape=(n,), dtype=jnp.float32)

# Add a gentle trend to force differencing utility
n = 140
trend = jnp.linspace(0.0, 6.0, n).astype(jnp.float32)
y = trend + _randn(n, scale=0.5, seed=9)

model = ARIMA(order=(1, 1, 1), include_mean=True, method="CSS")
model.fit(y)

res = model.predict(h=10)
mean = jnp.asarray(res["mean"])

print("Forecast shape:", mean.shape)
print("Forecast values:", mean)
print("All finite:", bool(jnp.all(jnp.isfinite(mean))))

assert mean.shape == (10,)
assert jnp.all(jnp.isfinite(mean)), "Mean forecast after inverse differencing must be finite"

print("\nFit + predict with differencing: OK")

## Forecast (Stateless)

The `forecast()` method fits and predicts in a single call without persisting any model state. This is convenient for cross-validation and batch evaluation.

In [ ]:
y = _randn(120, seed=123)
model = ARIMA(order=(1, 0, 1), include_mean=True, method="CSS")

# model is unfitted
assert getattr(model, "model_", None) is None

out = model.forecast(h=6, y=y)

# Still unfitted after a stateless forecast
assert getattr(model, "model_", None) is None
assert "mean" in out and out["mean"].shape == (6,)

print("Forecast mean:", out["mean"])
print("\nStateless forecast: OK")

## Examining Fit Diagnostics

The current ARIMA implementation does not store in-sample fitted values directly. Instead, inspect residuals, innovations, coefficient vectors, and the optimizer success flag.

In [ ]:
# Fit an ARIMA(2,1,0) model on trended data
n = 100
trend = jnp.linspace(0.0, 5.0, n).astype(jnp.float32)
y = trend + _randn(n, scale=0.3, seed=42)

model = ARIMA(order=(2, 1, 0), include_mean=True, method="CSS")
model.fit(y)

# Access diagnostics exposed by the current fit payload
residuals = jnp.asarray(model.model_["residuals"])
innovations = jnp.asarray(model.model_["innovations"])
coef = jnp.asarray(model.model_["coef"])
print("Residuals shape:", residuals.shape)
print("Innovations shape:", innovations.shape)
print("Coefficient vector:", coef)
print("ARIMA metadata:", model.model_["arma"])
print("Optimization success:", bool(model.model_["success"]))
print("Residual variance (sigma2):", model.model_["sigma2"])

# Residual arrays may include leading non-finite values from initialization
n_nonfinite = int(jnp.sum(~jnp.isfinite(residuals)))
print("Non-finite residual entries:", n_nonfinite)

# Edge Cases and Limitations

**Edge Cases:**
1. **Short series**: `fit` raises `ValueError` if series is too short for (p, d, q)
2. **Non-finite early residuals**: leading residual entries can be non-finite because CSS conditions on the initial recursion state

**Guarantees / Design Choices:**
1. Stationarity/invertibility is enforced by construction via PACF transforms
2. Differencing inversion is exact discrete integration

**Limitations:**
1. Fixed-order ARIMA only; model order selection is manual
2. CSS objective differs from exact MLE → parameters may differ vs Kalman fits
3. MA terms can make CSS optimization more sensitive to initialization/hyperparams